# TEKNOFEST 2026 — Variant Pathogenicity Classifier
## Phases 4 → 6  (Training · Feature Selection · Stacking Ensemble)

Colab **driver** for the modelling half of the pipeline. Phases 0–3 already produced the
materialised, leakage-free feature matrices and the persisted CV folds; every heavy routine
here lives in the importable `src/` package, so the notebook only orchestrates.

**Inputs (from Phase 0–3):**
- `data/processed/development_features.parquet` — `(3511, 522)` features + `Label` + `source`, **`NaN` preserved**.
- `data/processed/canary_features.parquet` — `(291, 522)`, the 10% hold-out touched **once** at the very end.
- `outputs/folds/fold_indices.json` — the 5-fold `StratifiedGroupKFold` indices (group = variant, stratify = label+source).

**Scope (stops exactly after Phase 6):**
- **Phase 4a** — default XGBoost / LightGBM baselines through the shared CV loop; per-source AUC/F1/MCC.
- **Phase 5** — **SHAP** importances + the **Null Importance Test** (≥50 label shuffles, `joblib`-parallel) + correlation pruning ⇒ the selected feature subset.
- **Phase 4b** — **Optuna** (`MedianPruner`) tuning of XGB / LGBM / CatBoost **on the selected subset only**.
- **Phase 4.6** — tier model **heads**: separate base models for **skeleton** (≥337 missing) vs **dense** (≤20 missing) rows.
- **Phase 6** — OOF collection ⇒ **Isotonic calibration** ⇒ diversity check ⇒ **rank-normalisation** ⇒ Level-1 **Logistic-Regression meta-learner** on the calibrated+ranked OOF **plus** raw meta-features (source flags + missingness tier).

> **Constraints honoured:** no `!pip install` (Colab ships everything); GPU params come **only** from `config.model_gpu_params()`; tree models receive `NaN` **natively** (no imputation); folds are **loaded**, never regenerated.

In [ ]:
# --- Environment setup -------------------------------------------------------
# Resolve the repo root (so ./data and ./outputs paths work) and put it on
# sys.path so `from src import ...` works whether launched from ./ or ./notebooks.
import os
import sys


def find_repo_root(start: str = None) -> str:
    p = start or os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(p, "data")) and os.path.isdir(os.path.join(p, "src")):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            break
        p = parent
    return start or os.getcwd()


ROOT = find_repo_root()
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd

from src import config as C
from src import cv_runner as cvr
from src import feature_selection as fsel
from src import stacking as stk

C.set_global_seed(C.SEED)
C.ensure_dirs()

# GPU parameters come EXCLUSIVELY from the centralized config (constraint #3).
USE_GPU = C.detect_gpu()
GPU_PARAMS = C.model_gpu_params(USE_GPU)

print("Repo root      :", os.getcwd())
print("Python         :", sys.version.split()[0])
print("GPU available  :", USE_GPU)
print("XGB  gpu params:", GPU_PARAMS["xgboost"])
print("LGBM gpu params:", GPU_PARAMS["lightgbm"])
print("Cat  gpu params:", GPU_PARAMS["catboost"])

## Phase 4a — Load Matrices, Folds & Default-Model Baselines

`cv_runner.load_development_matrix()` splits `Label`/`source` off the materialised parquet and
returns `X` with **`NaN` preserved** (`float32`, no imputation). `load_folds()` reads the **exact**
`fold_indices.json` from Phase 0.4 — folds are never regenerated.

The run-knobs below control the (expensive) Phase-5/4b budgets. The plan calls for **≥50** null
shuffles and **300** Optuna trials; lower them for a quick dry-run, then restore for the final run.

`scale_pos_weight` is computed **at runtime** as `count_benign / count_pathogenic ≈ 0.37` — Pathogenic
is the *majority* (≈74%), so this correctly **down-weights** it (plan 4.2; never hardcoded).

In [ ]:
# --- Load the Phase-3 matrices + the persisted folds -------------------------
X, y, source = cvr.load_development_matrix()      # NaN preserved, no Label/source in X
X_canary, y_canary, source_canary = cvr.load_canary_matrix()
folds = cvr.load_folds()                          # exact StratifiedGroupKFold indices

ALL_FEATURES = cvr.feature_columns(X)
SPW = cvr.compute_scale_pos_weight(y)             # ~0.37 (benign / pathogenic)

# ---- Run budgets (plan: >=50 shuffles, 300 trials) --------------------------
N_NULL_SHUFFLES = 50      # Phase 5.2 null importance test
N_OPTUNA_TRIALS = 300     # Phase 4b per-model search
SHAP_MAX_SAMPLES = 2000   # rows sampled per fold for SHAP
SHAP_TOP_N = 200          # keep null-survivors that are also top-N by mean|SHAP|

print(f"development X = {X.shape} | canary X = {X_canary.shape} | folds = {len(folds)}")
print(f"pos_rate = {y.mean():.4f} | scale_pos_weight = {SPW:.4f} | n_features = {len(ALL_FEATURES)}")
print("per-source rows:", pd.Series(source).value_counts().to_dict())
# Sanity: the 5 validation folds must exactly partition the development set.
val_union = sorted(sum([f["val"] for f in folds], []))
assert val_union == list(range(len(X))), "folds must partition development"
print("OK | folds partition development; NaN preserved:", bool(X.isna().any().any()))

In [ ]:
# --- Phase 4a: default XGBoost / LightGBM baselines on ALL features ----------
# These quick (untuned) models (a) give a reference CV score and (b) supply the
# fold models that SHAP importance (Phase 5.5) is computed from. Each learner
# factory bakes in the GPU params from config + the runtime scale_pos_weight,
# and receives NaN natively with early stopping on the val fold.
xgb_default_factory = cvr.make_xgb_learner(y=y, use_gpu=USE_GPU)
lgbm_default_factory = cvr.make_lgbm_learner(y=y, use_gpu=USE_GPU)

res_xgb_default = cvr.run_cv(xgb_default_factory, X, y, source, folds, name="xgb_default")
res_lgbm_default = cvr.run_cv(lgbm_default_factory, X, y, source, folds, name="lgbm_default")

for r in (res_xgb_default, res_lgbm_default):
    print(f"\n{r.name}: {r.summary()}")
    print("  per-source OOF AUC:", {k: round(v, 4) for k, v in r.per_source_oof_auc.items()})

## Phase 5 — Feature Selection (run **before** Optuna)

Three complementary filters aggressively prune the 522 features down to the true signals:

1. **Null Importance Test (5.2, gold standard)** — train a default GBM on the real labels, then
   on **≥50 label-shuffled** copies (`joblib`-parallel, CPU LightGBM so processes don't contend
   for the GPU). A feature survives only if its real gain importance exceeds the **95th percentile**
   of its null distribution — i.e. it does more than fit noise.
2. **SHAP importance (5.5)** — mean |SHAP| over the default-XGB fold models, each explained on its
   own validation rows (more reliable than `gain` for correlated features). Used as the final
   arbiter: we keep null-survivors that are also **top-`SHAP_TOP_N`** by SHAP.
3. **Correlation pruning (5.4)** — for `|Pearson| > 0.95` pairs, drop the one less point-biserially
   correlated with the label.

Outputs: `outputs/feature_selection/null_importance_results.csv`,
`shap_importance_ranking.csv`, `selected_features.json`, `final_feature_list.txt`.

In [ ]:
# --- Phase 5.5: SHAP importance from the default-XGB fold models -------------
# (mean |SHAP| averaged across folds, each explained on its own validation rows)
shap_imp = fsel.shap_importance_from_cv(res_xgb_default, X, folds, max_samples=SHAP_MAX_SAMPLES)
print("top-15 features by mean|SHAP|:")
print(shap_imp.head(15).round(4).to_string())

In [ ]:
# --- Phase 5.2 + 5.4: Null Importance Test + correlation pruning -------------
# NaN-native CPU LightGBM gain importances; joblib fans the >=50 shuffles across cores.
importance_fn = fsel.make_lgbm_importance_fn(n_estimators=400, n_jobs=1)

fs = fsel.run_feature_selection(
    importance_fn, X, y,
    n_shuffles=N_NULL_SHUFFLES,
    n_jobs=-1,
    correlation_threshold=0.95,
    shap_importance=shap_imp,   # 5.5 arbiter
    shap_top_n=SHAP_TOP_N,
    save=True,
)

SELECTED = fs.selected
print(f"\nSelected {len(SELECTED)} / {len(ALL_FEATURES)} features "
      f"(null-survivors={fs.meta['n_null_survivors']}, dropped-correlated={fs.dropped_correlated}).")
print("null-importance top-10 by null_score:")
print(fs.null_importance.head(10)[["feature", "actual_importance", "null_p95", "null_score", "keep"]]
      .round(3).to_string(index=False))

## Phase 4b — Optuna Tuning (`MedianPruner`) on the Selected Subset

Now that the feature set is pruned, run the expensive search **only** on `SELECTED`. Each study
maximises **mean CV AUC** across the persisted folds, with `MedianPruner(n_startup_trials=20,
n_warmup_steps=10)` killing unpromising trials early (the objective `trial.report()`s the running
mean AUC per fold). The search spaces are exactly the plan's (4.3 / 4.4 / 4.5). Per-trial estimator
count is capped for speed; the winning params are refit at full `n_estimators` (2000) with early
stopping in the next cell.

In [ ]:
# --- Phase 4b: tune XGB / LGBM / CatBoost on SELECTED features ---------------
best_xgb, study_xgb = cvr.tune_model(
    "xgb", X, y, folds, SELECTED, n_trials=N_OPTUNA_TRIALS, use_gpu=USE_GPU)
best_lgbm, study_lgbm = cvr.tune_model(
    "lgbm", X, y, folds, SELECTED, n_trials=N_OPTUNA_TRIALS, use_gpu=USE_GPU)
best_cat, study_cat = cvr.tune_model(
    "catboost", X, y, folds, SELECTED, n_trials=N_OPTUNA_TRIALS, use_gpu=USE_GPU)

print("\nBest params:")
print("  XGB     :", best_xgb)
print("  LGBM    :", best_lgbm)
print("  CatBoost:", best_cat)

## Phase 4 (tuned base models) + Phase 4.6 (missingness heads)

Train the five Level-0 base learners through the shared CV loop, each on the `SELECTED` features,
collecting leakage-free **OOF** probabilities (full development length) **and** fold-averaged
**canary** predictions (`X_extra`) for the final stacked evaluation:

- **`XGB_tuned`, `LGBM_tuned`, `CatBoost_tuned`** — the Optuna winners refit at full depth.
- **`Skeleton_head`** — trained **only** on skeleton rows (`is_skeleton_row == 1`, ≥337 missing) using
  the EK + AA + missingness-meta subset, yet predicting OOF for **every** validation row.
- **`Dense_head`** — trained **only** on dense rows (`is_dense_row == 1`, ≤20 missing) on the full subset.

Training each head on its own tier prevents the dense-row model from being distorted by skeleton-row
gradients and vice-versa, while still giving the meta-learner a dense, tier-specialised OOF column.

In [ ]:
# --- Tuned base models (refit at full n_estimators=2000, early stopping) -----
xgb_tuned_factory = cvr.make_xgb_learner(best_xgb, y=y, use_gpu=USE_GPU)
lgbm_tuned_factory = cvr.make_lgbm_learner(best_lgbm, y=y, use_gpu=USE_GPU)
cat_tuned_factory = cvr.make_catboost_learner(best_cat, y=y, use_gpu=USE_GPU)

res_xgb = cvr.run_cv(xgb_tuned_factory, X, y, source, folds, name="XGB_tuned",
                     feature_cols=SELECTED, X_extra=X_canary)
res_lgbm = cvr.run_cv(lgbm_tuned_factory, X, y, source, folds, name="LGBM_tuned",
                      feature_cols=SELECTED, X_extra=X_canary)
res_cat = cvr.run_cv(cat_tuned_factory, X, y, source, folds, name="CatBoost_tuned",
                     feature_cols=SELECTED, X_extra=X_canary)

# --- Phase 4.6 missingness heads --------------------------------------------
skeleton_mask = (X["is_skeleton_row"] == 1).to_numpy()
dense_mask = (X["is_dense_row"] == 1).to_numpy()
skeleton_features = cvr.tier_feature_cols(SELECTED, "skeleton")   # EK + AA + missingness meta
dense_features = cvr.tier_feature_cols(SELECTED, "dense")         # full selected set
print(f"skeleton rows={int(skeleton_mask.sum())} (features={len(skeleton_features)}) | "
      f"dense rows={int(dense_mask.sum())} (features={len(dense_features)})")

res_skeleton = cvr.run_cv(cvr.make_xgb_learner(best_xgb, y=y, use_gpu=USE_GPU),
                          X, y, source, folds, name="Skeleton_head",
                          feature_cols=skeleton_features, train_row_mask=skeleton_mask,
                          X_extra=X_canary, verbose=False)
res_dense = cvr.run_cv(cvr.make_xgb_learner(best_xgb, y=y, use_gpu=USE_GPU),
                       X, y, source, folds, name="Dense_head",
                       feature_cols=dense_features, train_row_mask=dense_mask,
                       X_extra=X_canary, verbose=False)

base_results = [res_xgb, res_lgbm, res_cat, res_skeleton, res_dense]
for r in base_results:
    r.save(save_models=False)   # OOF + canary preds + metrics.json
    print(f"  {r.name:15s} OOF AUC={r.oof_metrics['auc']:.4f} | "
          f"per-source={ {k: round(v, 4) for k, v in r.per_source_oof_auc.items()} }")

## Phase 6 — Stacking Ensemble

`run_stacking` executes the plan's order end-to-end:

1. **OOF collection (6.1)** — concatenate the five base-model OOF columns (validated: equal length, no `NaN`).
2. **Isotonic calibration (6.2)** — one `IsotonicRegression(out_of_bounds='clip')` per model, fit on its
   own OOF; calibrators persisted to `outputs/models/calibrators/` for test-time reuse.
3. **Diversity check (6.3)** — pairwise Pearson correlation of calibrated OOF (want `< 0.95`).
4. **Rank-normalisation (6.3)** — `rankdata / n` per model to erase scale differences.
5. **Meta-features (6.4)** — calibrated+ranked OOF columns **plus** raw meta-features: source flags
   (`is_master/kanser/pah/cftr`) + ordinal `missingness_tier_encoded`.
6. **Meta-learner (6.4)** — Logistic Regression with nested CV over `C` (reusing the group folds),
   yielding the honest **stacked OOF** and a final model for test time.

The same fitted stack scores the **canary** base predictions — its **only** use, as the overfit
canary check before any submission.

In [ ]:
# --- Phase 6: calibrate -> rank-norm -> meta-learner -------------------------
# Level-0 OOF (development) and fold-averaged canary predictions, in a fixed order.
oof_dict = {r.name: r.oof for r in base_results}
canary_dict = {r.name: r.extra_pred for r in base_results}

stack = stk.run_stacking(
    oof_dict, y, X,
    source=source,
    folds=folds,                      # reuse group folds for honest nested meta-CV
    extra_pred_dict=canary_dict,      # score the canary with the fitted stack
    X_extra=X_canary,
    save=True,
)

print("\nCalibration ECE per base model (lower = better):")
print(stack.calibration.to_string(index=False))
print("\nCalibrated-OOF correlation (diversity, want < 0.95):")
print(stack.diversity.round(3).to_string())

In [ ]:
# --- Stacked results + the one-time canary check -----------------------------
from sklearn.metrics import roc_auc_score

best_base = max(base_results, key=lambda r: r.oof_metrics["auc"])
print("Base model OOF AUC:")
for r in sorted(base_results, key=lambda r: -r.oof_metrics["auc"]):
    print(f"  {r.name:15s} {r.oof_metrics['auc']:.4f}")

print(f"\nrank-average baseline OOF AUC : {stack.info['rank_average_baseline_auc']:.4f}")
print(f"STACKED meta-learner OOF AUC  : {stack.info['stacked_oof_auc']:.4f}  "
      f"(best single base = {best_base.name} @ {best_base.oof_metrics['auc']:.4f})")
print(f"meta-learner: LogisticRegression(C={stack.info['best_C']}), "
      f"{stack.info['n_meta_features']} meta-features")
print("per-source STACKED OOF AUC    :",
      {k: round(v, 4) for k, v in stack.info.get("stacked_oof_auc_per_source", {}).items()})

# Canary: touched exactly once. A gap < ~0.02 vs stacked OOF AUC = no CV overfit.
canary_auc = roc_auc_score(y_canary, stack.extra_pred)
print(f"\nCANARY stacked AUC (one-time): {canary_auc:.4f}  "
      f"| gap vs OOF = {abs(canary_auc - stack.info['stacked_oof_auc']):.4f}")

In [ ]:
# --- Artifacts summary (end of Phase 6) --------------------------------------
import os

print("Phase 4-6 complete. Persisted artifacts:\n")
expected = [
    os.path.join(C.FEATURE_SELECTION_DIR, "null_importance_results.csv"),
    os.path.join(C.FEATURE_SELECTION_DIR, "shap_importance_ranking.csv"),
    C.SELECTED_FEATURE_LIST_PATH,
    os.path.join(C.FEATURE_SELECTION_DIR, "final_feature_list.txt"),
    os.path.join(C.OOF_DIR, "XGB_tuned_oof.npy"),
    os.path.join(C.OOF_DIR, "LGBM_tuned_oof.npy"),
    os.path.join(C.OOF_DIR, "CatBoost_tuned_oof.npy"),
    os.path.join(C.OOF_DIR, "Skeleton_head_oof.npy"),
    os.path.join(C.OOF_DIR, "Dense_head_oof.npy"),
    os.path.join(C.CALIBRATORS_DIR, "XGB_tuned_isotonic.pkl"),
    os.path.join(C.STACKING_DIR, "meta_learner.pkl"),
    os.path.join(C.STACKING_DIR, "stacked_oof.npy"),
    os.path.join(C.STACKING_DIR, "stacking_metrics.json"),
]
for p in expected:
    print(f"  [{'OK ' if os.path.exists(p) else 'MISS'}] {p}")

print("\nNext (out of scope here): Phase 7 threshold optimisation (global + per-source),\n"
      "Phase 8 SHAP/calibration report, Phase 9 ablations. Stopping exactly after Phase 6.")